In [4]:
import pandas as pd

In [24]:
fund_master = pd.read_csv("../data/raw/01_fund_master.csv")
nav_history = pd.read_csv("../data/raw/02_nav_history.csv")

In [25]:
fund_master.columns.tolist()
nav_history.columns.tolist()

['amfi_code', 'date', 'nav']

In [10]:
fund_master.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [11]:
print(fund_master["fund_house"].unique())

['SBI Mutual Fund' 'HDFC Mutual Fund' 'ICICI Prudential MF'
 'Nippon India MF' 'Kotak Mahindra MF' 'Axis Mutual Fund'
 'Aditya Birla Sun Life MF' 'UTI Mutual Fund' 'Mirae Asset MF'
 'DSP Mutual Fund']


In [12]:
print(fund_master["category"].unique())

['Equity' 'Debt']


In [17]:
print(fund_master["sub_category"].unique())

['Large Cap' 'Small Cap' 'Gilt' 'Mid Cap' 'Short Duration' 'Value'
 'Liquid' 'Index/ETF' 'Flexi Cap' 'Index' 'Large & Mid Cap' 'ELSS']


In [18]:
print(fund_master["risk_category"].unique())

['Moderate' 'Very High' 'Low' 'High' 'Moderately High']


In [21]:
fund_codes=set(fund_master["scheme_name"])

In [26]:
print(fund_master.columns.tolist())
print(nav_history.columns.tolist())

['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']
['amfi_code', 'date', 'nav']


In [27]:
fund_codes = set(fund_master["amfi_code"])

nav_codes = set(nav_history["amfi_code"])

missing_codes = fund_codes - nav_codes

print("Missing AMFI Codes:", len(missing_codes))
print(missing_codes)

Missing AMFI Codes: 0
set()


In [28]:
print("Fund Master Unique Codes:", fund_master["amfi_code"].nunique())

print("NAV History Unique Codes:", nav_history["amfi_code"].nunique())

print("Missing Codes:", len(missing_codes))

Fund Master Unique Codes: 40
NAV History Unique Codes: 40
Missing Codes: 0


Day-2 

Load

In [30]:
nav_history = pd.read_csv("../data/raw/02_nav_history.csv")

Convert dates

In [31]:
nav_history["date"] = pd.to_datetime(nav_history["date"])

Sort

In [32]:
nav_history = nav_history.sort_values(
    ["amfi_code","date"]
)

Remove duplicates

In [33]:
nav_history = nav_history.drop_duplicates()

Filling missing nav

In [34]:
nav_history["nav"] = nav_history.groupby(
    "amfi_code"
)["nav"].ffill()

Validate nav

In [35]:
bad_nav = nav_history[
    nav_history["nav"] <= 0
]

In [37]:
nav_history.to_csv(
    "../data/processed/nav_history_clean.csv",
    index=False
)

In [39]:
investor_transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")

In [40]:
print(investor_transactions.columns.tolist())

['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']


Standardize transaction types

In [42]:
investor_transactions["transaction_type"] = (
    investor_transactions["transaction_type"]
    .str.upper()
)

validate amount

In [43]:
investor_transactions = investor_transactions["amount_inr"] > 0

Fix date

In [54]:
print(
    investor_transactions["transaction_type"]
    .unique()
)

['SIP' 'REDEMPTION' 'LUMPSUM']


In [55]:
bad_amounts = investor_transactions[
    investor_transactions["amount_inr"] <= 0
]

print(len(bad_amounts))

0


In [56]:
print(
    investor_transactions["kyc_status"]
    .unique()
)

['Verified' 'Pending']


In [65]:
investor_transactions.to_csv(
    "../data/processed/investor_transactions_clean.csv",
    index=False
)

In [59]:
scheme_performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

In [61]:
print(scheme_performance.columns.tolist())

['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


In [62]:
scheme_performance["return_1yr_pct"] = pd.to_numeric(
    scheme_performance["return_1yr_pct"],
    errors="coerce"
)

In [64]:
bad_expense = scheme_performance[
    (scheme_performance["expense_ratio_pct"] < 0.1)
    |
    (scheme_performance["expense_ratio_pct"] > 2.5)
]

In [66]:
scheme_performance.to_csv(
    "../data/processed/scheme_performance_clean.csv",
    index=False
)

In [67]:
from sqlalchemy import create_engine

engine = create_engine(
    "sqlite:///bluestock_mf.db"
)